In [2]:
!unzip kaggle.zip

Archive:  kaggle.zip
  inflating: healthcare-dataset-stroke-data.csv  


In [3]:
import pandas as pd
import numpy as np

# Load the newly extracted CSV file
# (If your file name is slightly different, change the text inside the quotes to match it exactly!)
dataset = pd.read_csv('healthcare-dataset-stroke-data.csv')

# Look at the first 5 rows of your data
dataset.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
# Check which columns have missing values
dataset.isnull().sum()

,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,201


In [5]:
# Fill missing BMI values with the mean (average) of the BMI column
dataset['bmi'] = dataset['bmi'].fillna(dataset['bmi'].mean())

# Verify that there are no more missing values
dataset.isnull().sum()

,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,0


In [6]:
from sklearn.preprocessing import LabelEncoder

# Set up the encoder
le = LabelEncoder()

# List of columns that have text
text_columns = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

# Convert text to numbers for each column
for col in text_columns:
    dataset[col] = le.fit_transform(dataset[col])

# Let's see our converted dataset!
dataset.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,1,67.0,0,1,1,2,1,228.69,36.600000,1,1
1,51676,0,61.0,0,0,1,3,0,202.21,28.893237,2,1
2,31112,1,80.0,0,1,1,2,0,105.92,32.500000,2,1
3,60182,0,49.0,0,0,1,2,1,171.23,34.400000,3,1
4,1665,0,79.0,1,0,1,3,0,174.12,24.000000,2,1


In [7]:
# Separate features (X) and target variable (y)
# We drop 'id' and 'stroke' from X
X = dataset.drop(columns=['id', 'stroke'])
y = dataset['stroke']

# Split the dataset into 80% Training data and 20% Testing data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (4088, 10)
Testing features shape: (1022, 10)


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ==================== MODEL 1: Logistic Regression ====================
print("--- Training Model 1: Logistic Regression ---")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Make predictions
lr_predictions = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_predictions)
print(f"Logistic Regression Accuracy: {lr_accuracy * 100:.2f}%\n")


# ==================== MODEL 2: Random Forest ====================
print("--- Training Model 2: Random Forest ---")
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
rf_predictions = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)
print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%\n")


# Let's see a detailed report of our best performing model
print("--- Detailed Performance Report (Random Forest) ---")
print(classification_report(y_test, rf_predictions))

--- Training Model 1: Logistic Regression ---
Logistic Regression Accuracy: 93.93%

--- Training Model 2: Random Forest ---
Random Forest Accuracy: 93.84%

--- Detailed Performance Report (Random Forest) ---
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       960
           1       0.00      0.00      0.00        62

    accuracy                           0.94      1022
   macro avg       0.47      0.50      0.48      1022
weighted avg       0.88      0.94      0.91      1022



In [9]:
# Train a new Random Forest Model that handles imbalanced data automatically
rf_balanced = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_balanced.fit(X_train, y_train)

# Make predictions on the test data
rf_bal_predictions = rf_balanced.predict(X_test)

# Let's print the new performance report!
print("--- New and Improved Performance Report (Balanced Random Forest) ---")
print(classification_report(y_test, rf_bal_predictions))
print(classification_report(y_test, rf_bal_predictions))

--- New and Improved Performance Report (Balanced Random Forest) ---
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       960
           1       0.00      0.00      0.00        62

    accuracy                           0.94      1022
   macro avg       0.47      0.50      0.48      1022
weighted avg       0.88      0.94      0.91      1022

              precision    recall  f1-score   support

           0       0.94      1.00      0.97       960
           1       0.00      0.00      0.00        62

    accuracy                           0.94      1022
   macro avg       0.47      0.50      0.48      1022
weighted avg       0.88      0.94      0.91      1022



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [10]:
# 1. Install the imbalanced-learn library if not already installed
!pip install imbalanced-learn -q

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 2. Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# Check the new distribution of classes (they should now be equal)
print("Original training classes:", y_train.value_counts().to_dict())
print("Balanced training classes (After SMOTE):", y_train_balanced.value_counts().to_dict())
print("\n--- Training Model with SMOTE Balanced Data ---")

# 3. Train the Random Forest on the balanced data
rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_balanced, y_train_balanced)

# 4. Make predictions
rf_smote_predictions = rf_smote.predict(X_test)

# 5. Print the new performance report!
print("\n--- Final Performance Report (Random Forest with SMOTE) ---")
print(classification_report(y_test, rf_smote_predictions))

Original training classes: {0: 3901, 1: 187}
Balanced training classes (After SMOTE): {0: 3901, 1: 3901}

--- Training Model with SMOTE Balanced Data ---

--- Final Performance Report (Random Forest with SMOTE) ---
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       960
           1       0.17      0.16      0.17        62

    accuracy                           0.90      1022
   macro avg       0.56      0.56      0.56      1022
weighted avg       0.90      0.90      0.90      1022

